# Benchmark YouTube Maroc — Analyse de Marché (25 créateurs)
Comprendre les tendances de consommation au Maroc

**Structure du notebook :** 
`Regrouper les csv de tout les createurs de contenus dans un seul csv` →
`Nettoyage (supprimer colonnes vides, gérer les NaN)` →
`Feature engineering temporel (extract heure/jour/mois)` →
`Calcul engagement rate` →
`Classification thématique (à discuter — NLP ou manuelle ?)` →
`Analyse comparative par créateur et par thème` →
`Visualisations (meilleure heure, meilleur type de contenu, durée optimale)` →

## 1- Regrouper les csv de tout les createurs de contenus dans un seul csv

In [1]:
import pandas as pd
import glob

In [2]:
files = glob.glob("src/data*.csv")
df_list = []

for file in files:
    df = pd.read_csv(file, encoding="utf-8-sig")
    df_list.append(df)

# Fusionner tous les dataframes
df= pd.concat(df_list, ignore_index=True)

# Sauvegarder le dataset final
df.to_csv("df_benchmark.csv", index=False)

print("Fusion terminée !")
print("Nombre total de lignes :", df.shape[0])
print("Nombre total de colonnes :", df.shape[1])

Fusion terminée !
Nombre total de lignes : 9449
Nombre total de colonnes : 30


## 2- Nettoyage (supprimer colonnes vides, gérer les NaN)

In [3]:
df.head(5)

,position,channelId,channelTitle,videoId,publishedAt,publishedAtSQL,videoTitle,videoDescription,tags,videoCategoryId,...,licensedContent,hasPaidProductPlacement,locationDescription,latitude,longitude,viewCount,likeCount,dislikeCount,favoriteCount,commentCount
0,1,UCEsNqbwMdVFWmh1Thq13B4A,abdellah zarouk,zgAe3cqZVCE,2026-04-15T17:35:32Z,2026-04-15 17:35:32,طوب 5 | غرائب الجامعات المغربية 🤣🤣,kick: https://kick.com/zaroukabdellah For busi...,NaN,23.0,...,1.0,NaN,NaN,NaN,NaN,227474.0,7115.0,NaN,0.0,508.0
1,2,UCEsNqbwMdVFWmh1Thq13B4A,abdellah zarouk,u6avr_4SEd4,2026-03-23T17:16:32Z,2026-03-23 17:16:32,طوب 5 | أغرب فتاوى مغربية 🤣🤣,يسلط عبد الله زروق الضوء على فتاوى دينية مغربي...,"mi na3ima,boudbana,COMEDY,ABDELLAH ZAROUK,zaro...",23.0,...,1.0,NaN,NaN,NaN,NaN,300246.0,9126.0,NaN,0.0,860.0
2,3,UCEsNqbwMdVFWmh1Thq13B4A,abdellah zarouk,BxeiTEQ_Og4,2026-02-25T21:20:38Z,2026-02-25 21:20:38,طوب 5 | غرائب رياضية 🤣🤣,قناتي الخاصة بالألعاب: https://www.youtube.com...,"mi na3ima,boudbana,COMEDY,ABDELLAH ZAROUK,zaro...",23.0,...,1.0,NaN,NaN,NaN,NaN,162332.0,5114.0,NaN,0.0,256.0
3,4,UCEsNqbwMdVFWmh1Thq13B4A,abdellah zarouk,RO9N2JgaOeg,2026-02-17T22:24:47Z,2026-02-17 22:24:47,طوب 5 | أغرب مقالب مغربية 🤣🤣,يستعرض عبد الله زروق تطور مقالب وسائل التواصل ...,"mi na3ima,boudbana,COMEDY,ABDELLAH ZAROUK,zaro...",23.0,...,1.0,NaN,NaN,NaN,NaN,397151.0,12224.0,NaN,0.0,606.0
4,5,UCEsNqbwMdVFWmh1Thq13B4A,abdellah zarouk,4IDvvncwJvU,2026-01-20T18:08:10Z,2026-01-20 18:08:10,باريس لا تستحق 🤣🤣,قناتي الخاصة بالألعاب: https://www.youtube.com...,"mi na3ima,boudbana,COMEDY,ABDELLAH ZAROUK,zaro...",23.0,...,1.0,NaN,NaN,NaN,NaN,98611.0,4427.0,NaN,0.0,243.0


In [4]:
df.tail(5)

,position,channelId,channelTitle,videoId,publishedAt,publishedAtSQL,videoTitle,videoDescription,tags,videoCategoryId,...,licensedContent,hasPaidProductPlacement,locationDescription,latitude,longitude,viewCount,likeCount,dislikeCount,favoriteCount,commentCount
9444,122,UChOoQb2-j2adGHoi-OPxQQQ,Zizou Vlogs,3eQfAJMIwUQ,2020-11-13T17:30:19Z,2020-11-13 17:30:19,مشيت لجزيرة النساء وخلقنا السعادة معا تيتيز ال...,تابعوني على تطبيق YouNow كل يوم بث ⬇️ https://...,"Zizou vlogs,Hamada chroukate,Fayssal vlog,Fays...",22.0,...,1.0,NaN,NaN,NaN,NaN,396394.0,11596.0,NaN,0.0,668.0
9445,123,UChOoQb2-j2adGHoi-OPxQQQ,Zizou Vlogs,GbhCqxcNbxQ,2020-11-09T19:13:48Z,2020-11-09 19:13:48,حياة الليل فلاس فيغاس ديال المكسيك 😱 | Mexico ...,Aتابعوني على الانستغرام FOLLOW ME ON INSTAGRAM...,"Hassan diae vlogs,Mourad Mzouri vlogs,Aymane s...",22.0,...,1.0,NaN,NaN,NaN,NaN,284940.0,9214.0,NaN,0.0,638.0
9446,124,UChOoQb2-j2adGHoi-OPxQQQ,Zizou Vlogs,A-FIGii4ho8,2020-11-03T22:32:04Z,2020-11-03 22:32:04,Transmisión en vivo de Zizou Vlogs,NaN,NaN,22.0,...,1.0,NaN,NaN,NaN,NaN,0.0,4.0,NaN,0.0,0.0
9447,125,UChOoQb2-j2adGHoi-OPxQQQ,Zizou Vlogs,o9dYvTlsihE,2020-08-11T14:41:53Z,2020-08-11 14:41:53,درت مشروع ديال الهندية فالمكسيك بزيرو درهم شوف...,MY INSTAGRAM : zizouvlogs هدا الانستغرام ديال ...,"hassan diae vlogs,taha essou,niba tbib,hamada ...",22.0,...,1.0,NaN,NaN,NaN,NaN,624096.0,20901.0,NaN,0.0,2212.0
9448,126,UChOoQb2-j2adGHoi-OPxQQQ,Zizou Vlogs,iFhbXvOjogU,2020-07-06T16:59:44Z,2020-07-06 16:59:44,لقيت شارع فالمكسيك سميتو المغرب سماه مكسيكي بس...,‏INSTAGRAM :: @ZIZOUVLOGS ‏My second channel o...,"koulchi,bimo,abdellah zerrouk,taha essou,lala ...",22.0,...,1.0,NaN,NaN,NaN,NaN,195777.0,9300.0,NaN,0.0,800.0


In [5]:
df.head()
df.info()
df[['channelTitle']].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9449 entries, 0 to 9448
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   position                 9449 non-null   int64  
 1   channelId                9448 non-null   object 
 2   channelTitle             9448 non-null   object 
 3   videoId                  9449 non-null   object 
 4   publishedAt              9448 non-null   object 
 5   publishedAtSQL           9448 non-null   object 
 6   videoTitle               9448 non-null   object 
 7   videoDescription         8374 non-null   object 
 8   tags                     6539 non-null   object 
 9   videoCategoryId          9448 non-null   float64
 10  videoCategoryLabel       9448 non-null   object 
 11  topicCategories          6348 non-null   object 
 12  duration                 9448 non-null   object 
 13  durationSec              9449 non-null   int64  
 14  dimension               

,channelTitle
count,9448
unique,25
top,la3zawi family Two
freq,1741


In [6]:
df['publishedAtSQL']

0       2026-04-15 17:35:32
1       2026-03-23 17:16:32
2       2026-02-25 21:20:38
3       2026-02-17 22:24:47
4       2026-01-20 18:08:10
               ...         
9444    2020-11-13 17:30:19
9445    2020-11-09 19:13:48
9446    2020-11-03 22:32:04
9447    2020-08-11 14:41:53
9448    2020-07-06 16:59:44
Name: publishedAtSQL, Length: 9449, dtype: object

In [7]:
#verifier si j'ai des doublons
print("Nombre de doublons :", df.duplicated().sum())

Nombre de doublons : 0


In [8]:
# 2. NETTOYAGE : SUPPRESSION DES COLONNES INUTILES
cols_to_drop = ['dislikeCount', 'locationDescription', 'latitude', 
                'longitude', 'hasPaidProductPlacement', 'thumbnail_maxres']
# on utilise errors='ignore' au cas où elles seraient déjà supprimées
df = df.drop(columns=cols_to_drop, errors='ignore')
df.describe()


,position,videoCategoryId,durationSec,licensedContent,viewCount,likeCount,favoriteCount,commentCount
count,9449.000000,9448.000000,9449.000000,9126.0,9.448000e+03,9423.000000,9448.0,9424.000000
mean,355.523018,23.294877,1061.856704,1.0,8.958142e+05,34611.204287,0.0,1613.571626
std,369.974845,1.831371,722.103594,0.0,1.222544e+06,39821.015196,0.0,2597.231733
min,1.000000,1.000000,0.000000,1.0,0.000000e+00,0.000000,0.0,0.000000
25%,100.000000,22.000000,575.000000,1.0,2.197785e+05,9546.000000,0.0,399.750000
50%,228.000000,24.000000,954.000000,1.0,4.894920e+05,22957.000000,0.0,918.500000
75%,484.000000,24.000000,1452.000000,1.0,1.120512e+06,44440.500000,0.0,1880.250000
max,1741.000000,29.000000,3587.000000,1.0,1.914292e+07,667621.000000,0.0,60975.000000


In [9]:
# 3. NETTOYAGE : IMPUTATION DES VALEURS MANQUANTES
df['tags'] = df['tags'].fillna('')
df['videoDescription'] = df['videoDescription'].fillna('')
df['topicCategories'] = df['topicCategories'].fillna('Unknown')
df['likeCount'] = df['likeCount'].fillna(0)
df['commentCount'] = df['commentCount'].fillna(0)
df['viewCount'] = df['viewCount'].fillna(0)
df['durationSec'] = df['durationSec'].fillna(0)

# Traitement spécifique pour licensedContent (booléen vers binaire 0/1)
df['licensedContent'] = df['licensedContent'].fillna(0).astype(int)


---

## STEP 1 — Feature Engineering